# Corpus distributions

Read-only. Aggregates the three per-source files in `data/` into
`reports/corpus_distributions.md` and one figure in `figures/`. Modifies nothing.

Breakdowns: documents per year for every source, document type and ingest route for RIS,
court type and canton for Switzerland, document type and respondent for ECHR, and matched
keyword everywhere.

**Two cautions carried into the report itself.** Counts describe the *collection*, not the
courts — each source uses a different selection rule and a different search backend. And
`matched_keywords` records why a document was collected, not what it says: in RIS a decision
reached through a headnote link inherits that headnote's label, and `Suchworte` ANDs its
tokens rather than matching a phrase.

## 1. Load

Three per-source files, read-only. The only per-source logic is the date accessor: HUDOC
splits its dates across `judgementdate` (judgments), `decisiondate` (admissibility) and
`introductiondate` (communicated cases), and formats them `DD/MM/YYYY`, while RIS and
entscheidsuche use ISO.

In [1]:
# Corpus distributions — READ-ONLY. Aggregates data/ into reports/corpus_distributions.md
# and figures/. Does not modify any corpus file.
import json, re, collections
from datetime import date
from pathlib import Path

DATA    = Path("../data")
REPORTS = Path("../reports")
FIGURES = Path("../figures")
YEARS   = list(range(2000, 2026))

FILES = {"echr": "echr_parental_alienation.json",
         "ris":  "ris_parental_alienation.json",
         "swiss": "swiss_parental_alienation.json"}

_DMY = re.compile(r"^(\d{2})/(\d{2})/(\d{4})")

def yr(v):
    """Year from ISO (2011-05-17) or HUDOC's DD/MM/YYYY HH:MM:SS."""
    s = str(v or "")
    m = _DMY.match(s)
    if m:
        return int(m.group(3))
    try:
        return int(s[:4])
    except (TypeError, ValueError):
        return None

def load(k):
    return json.loads((DATA / FILES[k]).read_text(encoding="utf-8"))

echr, ris, swiss = load("echr"), load("ris"), load("swiss")
ris_te = [r for r in ris if r.get("dokumenttyp") == "Text"]
ris_rs = [r for r in ris if r.get("dokumenttyp") == "Rechtssatz"]

# per-source year accessor. ECHR: judgments carry judgementdate, admissibility
# decisions decisiondate, communicated cases only introductiondate.
def year_of(src, r):
    if src == "echr":
        return yr(r.get("judgementdate") or r.get("decisiondate") or r.get("introductiondate"))
    if src == "ris":
        return yr(r.get("entscheidungsdatum"))
    return yr(r.get("year") or r.get("Datum"))

OUT = []
def W(s=""): OUT.append(s)

def table(headers, rows, align=None):
    align = align or (["---"] + ["---:"] * (len(headers) - 1))
    W("| " + " | ".join(headers) + " |")
    W("|" + "|".join(align) + "|")
    for r in rows:
        W("| " + " | ".join(str(x) for x in r) + " |")
    W()

def counts_table(title, counter, label, top=None, total=None):
    W(f"**{title}**\n")
    items = counter.most_common(top)
    tot = total or sum(counter.values())
    table([label, "n", "%"],
          [(k if k not in (None, "") else "(none)", f"{v:,}", f"{100*v/tot:.1f}")
           for k, v in items])

print(f"echr {len(echr):,} | ris {len(ris):,} ({len(ris_rs)} Rechtssätze + {len(ris_te):,} decisions)"
      f" | swiss {len(swiss):,}")

echr 1,574 | ris 2,164 (38 Rechtssätze + 2,126 decisions) | swiss 2,031


## 2. Documents per year, all sources

In [9]:
# ---- 1. documents per year, all sources ------------------------------------
per_year = {}
for src, recs in [("echr", echr), ("ris", ris_te), ("swiss", swiss)]:
    per_year[src] = collections.Counter(year_of(src, r) for r in recs)

W("# Corpus distributions\n")
W(f"*Generated {date.today().isoformat()} by `src/corpus_distributions.ipynb` — read-only.*\n")
W("Counts describe **the collection**, not the courts. Each source was assembled under a "
  "different selection rule and a different search backend, so columns are not comparable "
  "with one another. See `reports/corpus_composition.md` for the selection rules.\n")
W("## 1. Documents per year\n")
W("RIS counts decisions only (the 38 Rechtssätze are undated in the decision sense — a "
  "headnote's date is that of the most recent decision applying it).\n")
rows = []
for y in YEARS:
    e, r_, s = per_year["echr"][y], per_year["ris"][y], per_year["swiss"][y]
    rows.append((y, f"{e:,}", f"{r_:,}", f"{s:,}", f"{e+r_+s:,}"))
tot = (sum(per_year["echr"][y] for y in YEARS), sum(per_year["ris"][y] for y in YEARS),
       sum(per_year["swiss"][y] for y in YEARS))
rows.append(("**total**", f"**{tot[0]:,}**", f"**{tot[1]:,}**", f"**{tot[2]:,}**",
             f"**{sum(tot):,}**"))
table(["year", "ECHR", "AT (decisions)", "CH", "all"], rows)
# account for every record not in the table above
for src, recs in [("echr", echr), ("ris", ris_te), ("swiss", swiss)]:
    undated = sum(1 for r in recs if year_of(src, r) is None)
    oor = sum(1 for r in recs
              if (y := year_of(src, r)) is not None and not (2000 <= y <= 2025))
    if undated or oor:
        by_type = collections.Counter(
            r.get("doctypebranch") or r.get("dokumenttyp") or "(n/a)"
            for r in recs if year_of(src, r) is None)
        W(f"- `{src}`: **{undated:,} undated**"
          + (f" (all {', '.join(f'{k}' for k in by_type)})" if len(by_type) == 1 else
             f" ({dict(by_type)})")
          + (f", {oor} outside 2000–2025" if oor else "") + ".")
W()
W("The undated ECHR records are **communicated cases** — applications notified to the "
  "respondent government but not yet decided. HUDOC exposes no judgment, decision or "
  "introduction date for them, so they cannot appear in any time series and are excluded "
  "from every per-year table below. They remain in the corpus and in the index: they are "
  "retrievable, they just cannot be dated.\n")
print("\n".join(OUT[-8:]))

| 2023 | 103 | 85 | 190 | 378 |
| 2024 | 63 | 69 | 203 | 335 |
| 2025 | 43 | 130 | 238 | 411 |
| **total** | **1,232** | **2,126** | **2,031** | **5,389** |

- `echr`: **342 undated** (all COMMUNICATEDCASES).

The undated ECHR records are **communicated cases** — applications notified to the respondent government but not yet decided. HUDOC exposes no judgment, decision or introduction date for them, so they cannot appear in any time series and are excluded from every per-year table below. They remain in the corpus and in the index: they are retrievable, they just cannot be dated.



## 3. ECHR breakdowns

In [3]:
# ---- 2. ECHR: document type, respondent, keyword ---------------------------
W("## 2. ECHR (HUDOC)\n")
W(f"{len(echr):,} documents, English, exact-phrase `fulltext` search on HUDOC.\n")
counts_table("By document type (`doctypebranch`)",
             collections.Counter(r.get("doctypebranch") for r in echr), "document type")
counts_table("By respondent state (top 15)",
             collections.Counter(r.get("respondent") for r in echr), "respondent", top=15)
counts_table("By matched keyword (a document may match several)",
             collections.Counter(k for r in echr for k in (r.get("matched_keywords") or [])),
             "keyword", total=len(echr))
W("Document type per year:\n")
dts = [d for d, _ in collections.Counter(r.get("doctypebranch") for r in echr).most_common()]
rows = []
for y in YEARS:
    sel = [r for r in echr if year_of("echr", r) == y]
    rows.append([y] + [sum(1 for r in sel if r.get("doctypebranch") == d) for d in dts] + [len(sel)])
table(["year"] + dts + ["total"], rows)
print("\n".join(OUT[-6:]))

| 2021 | 27 | 0 | 3 | 22 | 16 | 6 | 74 |
| 2022 | 32 | 0 | 4 | 13 | 15 | 1 | 65 |
| 2023 | 24 | 0 | 3 | 44 | 31 | 1 | 103 |
| 2024 | 28 | 0 | 1 | 25 | 7 | 2 | 63 |
| 2025 | 21 | 0 | 2 | 11 | 9 | 0 | 43 |



## 4. RIS breakdowns

Both the document type (Rechtssatz vs decision) and the ingest route, since the Austrian
corpus is built by two different searches.

In [4]:
# ---- 3. RIS: document type, ingest route, court, keyword -------------------
W("## 3. RIS (Austria)\n")
W(f"{len(ris):,} records. Two search routes; `Suchworte` is an **AND of tokens**, not a "
  "phrase, so a multi-word keyword matches documents containing all its words anywhere.\n")
counts_table("By document type", collections.Counter(r.get("dokumenttyp") for r in ris),
             "Dokumenttyp")
counts_table("By ingest route (`match_route`)",
             collections.Counter("+".join(sorted(r.get("match_route") or ["(none)"])) for r in ris),
             "route")
W("- `rechtssatz` — a headnote matched a search term\n"
  "- `rechtssatz-link` — a decision reached by following a matched headnote's links\n"
  "- `fulltext` — the decision's own text matched a search term\n")
counts_table("By court (decisions only)",
             collections.Counter(r.get("gericht") for r in ris_te), "Gericht")
counts_table("By matched keyword (decisions only; a decision may carry several)",
             collections.Counter(k for r in ris_te for k in (r.get("matched_keywords") or [])),
             "keyword", total=len(ris_te))
W("> `matched_keywords` records **why a document was collected**, not what it says. A decision "
  "reached through a headnote link inherits that headnote's label, and because `Suchworte` ANDs "
  "its tokens, a multi-word label does not imply the phrase occurs. Do not compute "
  "\"how many decisions mention X\" from this field — search the stored text instead.\n")
W("Decisions per year by ingest route:\n")
routes = ["fulltext", "rechtssatz-link"]
rows = []
for y in YEARS:
    sel = [r for r in ris_te if year_of("ris", r) == y]
    rows.append([y] + [sum(1 for r in sel if rt in (r.get("match_route") or [])) for rt in routes]
                + [len(sel)])
table(["year"] + routes + ["total"], rows)
print("\n".join(OUT[-6:]))

| 2021 | 81 | 14 | 95 |
| 2022 | 74 | 13 | 87 |
| 2023 | 65 | 20 | 85 |
| 2024 | 58 | 11 | 69 |
| 2025 | 101 | 29 | 130 |



## 5. Swiss breakdowns

In [5]:
# ---- 4. Swiss: canton, court type, keyword ---------------------------------
W("## 4. entscheidsuche.ch (Switzerland)\n")
W(f"{len(swiss):,} decisions, German-language only, exact-phrase (`match_phrase`) search over "
  "the extracted text of the published PDF.\n")
fed = sum(1 for r in swiss if r.get("canton") == "CH")
W(f"Federal: {fed:,} · cantonal: {len(swiss)-fed:,}\n")
counts_table("By court type", collections.Counter(r.get("court_type") for r in swiss),
             "court type", top=15)
cant = collections.Counter(r.get("canton") for r in swiss)
counts_table("By canton (`CH` = federal courts)", cant, "canton")
ALL26 = "AG AI AR BE BL BS FR GE GL GR JU LU NE NW OW SG SH SO SZ TG TI UR VD VS ZG ZH".split()
missing = [c for c in ALL26 if c not in cant]
odd = [c for c in cant if c not in ALL26 and c != "CH"]
W(f"Cantons represented: {sum(1 for c in ALL26 if c in cant)} of 26. "
  f"Absent: {', '.join(missing)}."
  + (f" Non-standard codes present: {', '.join(odd)}." if odd else "") + "\n")
counts_table("By matched keyword (a decision may carry several)",
             collections.Counter(k for r in swiss for k in (r.get("matched_keywords") or [])),
             "keyword", total=len(swiss))
W("Federal vs cantonal per year:\n")
rows = []
for y in YEARS:
    sel = [r for r in swiss if year_of("swiss", r) == y]
    f_ = sum(1 for r in sel if r.get("canton") == "CH")
    rows.append((y, f"{f_:,}", f"{len(sel)-f_:,}", f"{len(sel):,}"))
table(["year", "federal", "cantonal", "total"], rows)
print("\n".join(OUT[-6:]))

| 2021 | 39 | 129 | 168 |
| 2022 | 31 | 170 | 201 |
| 2023 | 37 | 153 | 190 |
| 2024 | 39 | 164 | 203 |
| 2025 | 55 | 183 | 238 |



## 6. Matched keyword by year

In [6]:
# ---- 5. keyword x year, per source -----------------------------------------
W("## 5. Matched keyword by year\n")
W("Read these as *collection* series, not usage series: they show when documents that were "
  "collected under a term were decided. For RIS the caveat in § 3 applies.\n")
for src, recs, label in [("echr", echr, "ECHR"), ("ris", ris_te, "RIS (decisions)"),
                         ("swiss", swiss, "Switzerland")]:
    kws = [k for k, _ in collections.Counter(
        k for r in recs for k in (r.get("matched_keywords") or [])).most_common()]
    if len(kws) > 8:
        kws = kws[:8]
    W(f"**{label}**\n")
    rows = []
    for y in YEARS:
        sel = [r for r in recs if year_of(src, r) == y]
        rows.append([y] + [sum(1 for r in sel if k in (r.get("matched_keywords") or []))
                           for k in kws])
    table(["year"] + kws, rows)
print("wrote keyword x year for 3 sources")

wrote keyword x year for 3 sources


## 7. Figure

In [7]:
# ---- 6. figure: documents per year, one panel per source -------------------
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

FIGURES.mkdir(exist_ok=True)
fig, axes = plt.subplots(1, 3, figsize=(13, 3.4), sharex=True)
panels = [("echr", "ECHR (HUDOC)", "#4C6EA8"),
          ("ris", "Austria (OGH)", "#8C5E3C"),
          ("swiss", "Switzerland", "#4F7A63")]
for ax, (src, title, colour) in zip(axes, panels):
    vals = [per_year[src][y] for y in YEARS]
    ax.bar(YEARS, vals, color=colour, width=0.75)
    ax.set_title(title, fontsize=10)
    ax.set_xlim(1999.3, 2025.7)
    ax.tick_params(labelsize=8)
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(axis="y", alpha=0.25, linewidth=0.6)
axes[0].set_ylabel("documents", fontsize=9)
fig.suptitle("Documents per year by source (collection counts, not court output)",
             fontsize=11, y=1.02)
fig.tight_layout()
path = FIGURES / "corpus_documents_per_year.png"
fig.savefig(path, dpi=200, bbox_inches="tight")
plt.close(fig)
W(f"## 6. Figure\n")
W(f"![Documents per year by source](../figures/{path.name})\n")
W(f"`figures/{path.name}` — note the y-axes are independent; the Swiss panel rises with "
  "publication coverage, not litigation.\n")
print("wrote", path)

wrote ../figures/corpus_documents_per_year.png


## 8. Write the report

In [8]:
# ---- 7. write the report ---------------------------------------------------
REPORTS.mkdir(exist_ok=True)
target = REPORTS / "corpus_distributions.md"
target.write_text("\n".join(OUT) + "\n", encoding="utf-8")
print(f"wrote {target}  ({len(OUT)} lines)")

wrote ../reports/corpus_distributions.md  (382 lines)
